# tonitrus.com — Discovery Notebook

**Pass 0 · Category extraction from nav HTML**  
**Pass 1 · PLP crawl → product list + pagination**  
**Pass 2 · Sample PDP requests**

Async-capable via `asyncio.run()` (Jupyter-safe with nest_asyncio).

In [2]:
# ── standard library ──────────────────────────────────────────────────────────
import asyncio, json, logging, random, re, time
from collections import defaultdict
from urllib.parse import urljoin, urlparse, urlencode
from curl_cffi.requests import AsyncSession


# ── third-party ────────────────────────────────────────────────────────────────
import aiohttp                        # async HTTP
from bs4 import BeautifulSoup         # HTML parsing

# ── Jupyter async compatibility ────────────────────────────────────────────────
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("nest_asyncio applied ✓")
except ImportError:
    print("nest_asyncio not found — install with: pip install nest_asyncio")


nest_asyncio applied ✓


In [3]:
# ── logging — DEBUG level so you see every request ────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("tonitrus")

# Silence chatty third-party loggers
for noisy in ("urllib3", "charset_normalizer", "aiohttp.client"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

log.info("Logging initialised at DEBUG level")


14:26:30  INFO      tonitrus  Logging initialised at DEBUG level


In [4]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
BASE_URL        = "https://www.tonitrus.com/?lang=eng"
ITEMS_PER_PAGE  = 10
CONCURRENCY     = 1          # number of parallel Camoufox pages
SAMPLE_CATS     = 5
SAMPLE_PAGES    = 3
REQUEST_TIMEOUT = 30          # seconds

# ── PROXY ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 33335
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def make_proxy(token: str | None = None) -> dict:
    """Return a Camoufox-compatible proxy dict with optional sticky token."""
    if token is None:
        token = f"{random.random():.16f}"
    user = f"{PROXY_USER}-session-{token}"
    log.debug("proxy session  user=%s", user)
    return {
        "server":   f"http://{PROXY_HOST}:{PROXY_PORT}",
        "username": user,
        "password": PROXY_PASS,
    }

log.info("Config ready  base=%s  concurrency=%d  items/page=%d",
         BASE_URL, CONCURRENCY, ITEMS_PER_PAGE)

14:26:31  INFO      tonitrus  Config ready  base=https://www.tonitrus.com/?lang=eng  concurrency=1  items/page=10


In [5]:
# ── async fetch helper ────────────────────────────────────────────────────────
from camoufox.async_api import AsyncCamoufox
from playwright.async_api import Page

async def fetch(page: Page, url: str) -> tuple[str, float, int]:
    """
    Fetch *url* using an existing Camoufox page.
    Returns (html_text, elapsed_seconds, content_length_bytes).
    """
    t0 = time.perf_counter()
    log.debug("GET  url=%s", url)
    try:
        resp = await page.goto(url, wait_until="domcontentloaded", timeout=90000)
        html    = await page.content()
        elapsed = time.perf_counter() - t0
        log.debug("OK   status=%d  len=%d  t=%.2fs", resp.status, len(html), elapsed)
        return html, elapsed, len(html.encode())
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        log.warning("FAIL url=%s  err=%s  t=%.2fs", url, exc, elapsed)
        raise


async def _health_check(page: Page) -> bool:
    log.info("Proxy health-check …")
    try:
        html, t, sz = await fetch(page, BASE_URL)
        ok = "tonitrus" in html.lower()
        log.info("Proxy health-check  PASS=%s  t=%.2fs  size=%d B", ok, t, sz)
        return ok
    except Exception as exc:
        log.error("Proxy health-check FAILED: %s", exc)
        return False

In [6]:
# ── PASS 0 — category extraction from nav HTML ────────────────────────────────
def extract_categories(soup: BeautifulSoup) -> list[dict]:
    """
    Walk the tonitrus navigation and return every leaf category as a dict:
      {section, brand, type, name, url}

    The nav has two structurally different section types:
      A) Server / Storage / Desktop PC / Laptops
         → brand tabs rendered as <li.nav-item.dropdown> with a
           <a.nav-link.dropdown-toggle> brand label, then leaf <a.nav-link>
      B) Netzwerk
         → brand context in <li.nav-item.d-lg-none> "X anzeigen" links,
           product-type dropdowns nested inside <div.tab-pane>
    """
    import re as _re
    results: list[dict] = []
    seen:    set[str]   = set()

    MAIN_SECTIONS = {"Server", "Storage", "Netzwerk", "Desktop PC", "Laptops"}
    SKIP_FRAGS    = {"leistungen", "unternehmen", "kontakt", "Konfigurator",
                     "impressum", "datenschutz", "sitemap", "newsletter",
                     "remarketing", "logistik", "nachhaltigkeit", "standorte"}

    def _clean(text: str) -> str:
        return _re.sub(r"\s*\(\s*-?\d+\s*\)\s*$", "", text).strip()

    def _ok(href: str) -> bool:
        if not href or href == "#":
            return False
        return not any(f in href for f in SKIP_FRAGS)

    def _add(section, brand, typ, name, href):
        full = urljoin(BASE_URL, href) if href.startswith("/") else href
        if full in seen:
            return
        seen.add(full)
        results.append(dict(section=section, brand=brand, type=typ,
                            name=_clean(name), url=full))

    for top_li in soup.select("li.nav-item.dropdown"):
        toggle = top_li.select_one("a.nav-link.dropdown-toggle")
        if not toggle:
            continue
        section = toggle.get_text(strip=True)
        if section not in MAIN_SECTIONS:
            continue

        log.debug("pass0  section=%s", section)

        # ── Strategy B: Netzwerk — brand tabs ────────────────────────────
        tab_panes = top_li.select("div.tab-pane")
        if tab_panes:
            for pane in tab_panes:
                # Brand name from "X anzeigen" mobile header
                brand_a   = pane.select_one("li.nav-item.d-lg-none > a")
                brand     = brand_a.get_text(strip=True).replace(" anzeigen", "").strip() if brand_a else ""

                # Product-type sub-dropdowns
                for type_li in pane.select("li.nav-item.dropdown"):
                    type_a   = type_li.select_one("a.categories-recursive-link")
                    prod_typ = type_a.get_text(strip=True) if type_a else ""
                    for leaf in type_li.select("ul.nav.lpxcollapse li.nav-item:not(.d-lg-none)"):
                        a = leaf.select_one("a[href]")
                        if a and _ok(a["href"]):
                            _add(section, brand, prod_typ, a.get_text(strip=True), a["href"])

                # Flat items directly in pane (no sub-dropdown)
                for flat_li in pane.select("li.nav-item:not(.dropdown):not(.d-lg-none)"):
                    a = flat_li.select_one("a.nav-link[href]")
                    if a and _ok(a["href"]) and "anzeigen" not in a.get_text():
                        _add(section, brand, "", a.get_text(strip=True), a["href"])
            continue

        # ── Strategy A: Server / Storage / Desktop PC / Laptops ──────────
        # Brand groups are separated by <li.nav-item.d-lg-none> "X anzeigen"
        current_brand = ""
        for child_li in top_li.select("li.nav-item"):
            # Brand separator
            anzeigen = child_li.select_one("a")
            if "d-lg-none" in child_li.get("class", []) and anzeigen and "anzeigen" in anzeigen.get_text():
                current_brand = anzeigen.get_text(strip=True).replace(" anzeigen", "").strip()
                log.debug("pass0    brand=%s", current_brand)
                continue

            # Sub-dropdown (product type level)
            type_a   = child_li.select_one("a.categories-recursive-link")
            if type_a:
                prod_typ = type_a.get_text(strip=True)
                for leaf in child_li.select("ul.nav.lpxcollapse li.nav-item:not(.d-lg-none)"):
                    a = leaf.select_one("a[href]")
                    if a and _ok(a["href"]):
                        _add(section, current_brand, prod_typ, a.get_text(strip=True), a["href"])
                continue

            # Flat leaf directly under the section dropdown
            a = child_li.select_one("a.nav-link[href]")
            if a and _ok(a["href"]) and "anzeigen" not in a.get_text():
                _add(section, current_brand, "", a.get_text(strip=True), a["href"])

    log.info("pass0  extracted %d unique categories", len(results))
    return results


log.info("extract_categories() defined")


14:26:35  INFO      tonitrus  extract_categories() defined


In [11]:
# ── PLP parser (product listing page) ─────────────────────────────────────────
def parse_plp(html: str, cat_url: str) -> tuple[list[dict], int]:
    """
    Parse one PLP page. Returns (products_list, total_items_on_category).
    total_items is 0 if not found (use -1 to signal error).
    """
    soup = BeautifulSoup(html, "lxml")
        # ── temporary debug: show what card selectors actually match ──────────
    log.debug("plp  div[itemprop=itemListElement] count : %d",
              len(soup.select("div[itemprop='itemListElement']")))
    log.debug("plp  article.product-list-item count     : %d",
              len(soup.select("article.product-list-item")))
    log.debug("plp  a.text-clamp-2 count                : %d",
              len(soup.select("a.text-clamp-2")))
    log.debug("plp  productlist-item-info               : %s",
              soup.select_one("div.productlist-item-info"))
    body = soup.select_one("body")
    log.debug("plp  body snippet:\n%s", body.get_text()[:500] if body else "NO BODY")
    
    products: list[dict] = []

    # ── total item count ──────────────────────────────────────────────────
    total = 0
    info_div = soup.select_one("div.productlist-item-info")
    if info_div:
        m = re.search(r"of\s+([\d\.,]+)", info_div.get_text())
        if m:
            total = int(m.group(1).replace(".", "").replace(",", ""))
    log.debug("plp  url=%s  total=%d", cat_url, total)

        # ── product cards ─────────────────────────────────────────────────────
    for card in soup.select("div[itemprop='itemListElement']"):
        try:
            # URL — first <a href> in the card
            a_tags = card.select("a[href]")
            if not a_tags:
                continue
            url    = a_tags[0]["href"]
            handle = urlparse(url).path.lstrip("/")

            # MPN — handle without trailing _N suffix
            mpn = re.sub(r"_\d+$", "", handle)

            # Name — link text of first <a>
            name = a_tags[0].get_text(strip=True) or mpn

            # Price
            price_tag = card.select_one("meta[itemprop='price']")
            price_min = price_tag["content"] if price_tag else None

            min_tag = card.select_one("meta[itemprop='minPrice']")
            max_tag = card.select_one("meta[itemprop='maxPrice']")
            price_min = min_tag["content"] if min_tag else price_min
            price_max = max_tag["content"] if max_tag else price_min

            # Availability — not in card, default to Unknown, PDP will enrich
            avail = "Unknown"

            # CTO detection
            is_cto = "CTO" in url.upper() or "CTO" in name.upper()

            products.append(dict(
                handle=handle,
                name=name,
                brand="",        # not available at card level
                mpn=mpn,
                ean="",          # not available at card level
                price_min=price_min,
                price_max=price_max,
                availability=avail,
                is_cto=is_cto,
                pdp_required=is_cto,   # only CTOs strictly need PDP; others have MPN
                url=url,
            ))
        except Exception as exc:
            log.debug("plp  card parse error: %s", exc)
            continue

    log.debug("plp  parsed %d products from page", len(products))
    return products, total


log.info("parse_plp() defined")


14:30:27  INFO      tonitrus  parse_plp() defined


In [8]:
# ── PDP parser (product detail page — lightweight, for enrichment) ─────────────
def parse_pdp_variants(html: str) -> list[dict]:
    """
    Extract variation rows (NEW / REFURBISHED) from a PDP.
    Returns list of dicts:
      {condition, ref_id, sku, price_gross, price_net, stock, is_oos}
    """
    soup     = BeautifulSoup(html, "lxml")
    variants = []

    for row in soup.select("div.variation-wrapper, div.variation-w"):
        try:
            cond_tag = row.select_one("span.variation-label, dt")
            cond     = cond_tag.get_text(strip=True) if cond_tag else "UNKNOWN"

            ref_tag = row.select_one("input[name='variationID'], input[name='variation']")
            ref_id  = ref_tag["value"] if ref_tag else ""

            sku_tag = row.select_one("span.variation-sku, [itemprop='sku']")
            sku     = sku_tag.get_text(strip=True) if sku_tag else ""

            price_gross = None
            price_net   = None
            pg = row.select_one("meta[itemprop='price'], span.price-gross")
            if pg:
                price_gross = pg.get("content") or pg.get_text(strip=True)
            pn = row.select_one("span.price-net, span.tax-info")
            if pn:
                price_net = pn.get_text(strip=True)

            stock_tag = row.select_one("span.stock-count, [itemprop='inventoryLevel']")
            stock_raw = stock_tag.get_text(strip=True) if stock_tag else "0"
            stock     = int(re.sub(r"[^\d]", "", stock_raw) or "0")
            is_oos    = stock == 0 or bool(row.select_one("span.oos-label, .out-of-stock"))

            variants.append(dict(
                condition=cond,
                ref_id=ref_id,
                sku=sku,
                price_gross=price_gross,
                price_net=price_net,
                stock=stock,
                is_oos=is_oos,
            ))
        except Exception as exc:
            log.debug("pdp  variant parse error: %s", exc)
            continue

    log.debug("pdp  found %d variants", len(variants))
    return variants


log.info("parse_pdp_variants() defined")


14:26:43  INFO      tonitrus  parse_pdp_variants() defined


In [12]:
async def discovery():
    t_start = time.perf_counter()

    # ── 0. proxy health-check + homepage ─────────────────────────────────
    async with AsyncCamoufox(
        headless=True,
        geoip=True,
        proxy=make_proxy(),
    ) as browser:
        page     = await browser.new_page()
        proxy_ok = await _health_check(page)
        if not proxy_ok:
            log.error("Proxy health-check failed — aborting")
            return

        log.info("pass0  extracting categories from nav HTML …")
        soup_home   = BeautifulSoup(await page.content(), "lxml")
        total_bytes = len((await page.content()).encode())
        await page.close()

    categories = extract_categories(soup_home)
    by_section  = defaultdict(list)
    for c in categories:
        by_section[c["section"]].append(c)
    log.info("pass0  %d categories across %d sections",
             len(categories), len(by_section))

    # ── 1. PLP crawl — one browser per category ───────────────────────────
    sample_cats  = categories[:SAMPLE_CATS]
    all_products: list[dict]  = []
    page_sizes:   list[int]   = []
    page_times:   list[float] = []
    cat_totals:   dict        = {}

    sem = asyncio.Semaphore(CONCURRENCY)

    async def crawl_category(cat: dict):
        nonlocal total_bytes
        async with sem:
            cat_token = f"{random.random():.16f}"
            async with AsyncCamoufox(
                headless=True,
                geoip=True,
                proxy=make_proxy(cat_token),
            ) as browser:
                page    = await browser.new_page()
                cat_url = cat["url"]

                # first page
                first_url = f"{cat_url}?lang=eng&af={ITEMS_PER_PAGE}"
                log.info("pass1  cat=%s  url=%s", cat["name"], first_url)
                try:
                    html, t, sz = await fetch(page, first_url)
                except Exception as exc:
                    log.warning("pass1  cat=%s  FAILED: %s", cat["name"], exc)
                    return

                page_sizes.append(sz)
                page_times.append(t)
                total_bytes += sz
                prods, total = parse_plp(html, first_url)
                cat_totals[cat_url] = total
                all_products.extend(prods)

                # additional sample pages
                if total > ITEMS_PER_PAGE and SAMPLE_PAGES > 1:
                    n_pages = min(SAMPLE_PAGES, -(-total // ITEMS_PER_PAGE))
                    for pg in range(2, n_pages + 1):
                        page_url = f"{cat_url}_s{pg}?lang=eng&af={ITEMS_PER_PAGE}"
                        log.info("pass1  page=%d  url=%s", pg, page_url)
                        try:
                            html_p, t_p, sz_p = await fetch(page, page_url)
                        except Exception as exc:
                            log.warning("pass1  page=%d  FAILED: %s", pg, exc)
                            break
                        page_sizes.append(sz_p)
                        page_times.append(t_p)
                        total_bytes += sz_p
                        prods_p, _ = parse_plp(html_p, page_url)
                        all_products.extend(prods_p)

                await page.close()

    await asyncio.gather(*[crawl_category(c) for c in sample_cats])

    # de-duplicate by URL
    seen_urls = set()
    deduped   = []
    for p in all_products:
        if p["url"] not in seen_urls:
            seen_urls.add(p["url"])
            deduped.append(p)
    all_products = deduped

    # ── 2. sample PDP ─────────────────────────────────────────────────────
    pdp_candidates = [p for p in all_products if p.get("pdp_required")]
    pdp_sample     = None
    if pdp_candidates:
        async with AsyncCamoufox(
            headless=True,
            geoip=True,
            proxy=make_proxy(),
        ) as browser:
            pdp_page = await browser.new_page()
            log.info("pass2  fetching sample PDP: %s", pdp_candidates[0]["url"])
            try:
                html_pdp, t_pdp, sz_pdp = await fetch(pdp_page, pdp_candidates[0]["url"])
                total_bytes += sz_pdp
                pdp_sample   = parse_pdp_variants(html_pdp)
                log.info("pass2  sample PDP variants=%d", len(pdp_sample))
            except Exception as exc:
                log.warning("pass2  PDP fetch FAILED: %s", exc)
            await pdp_page.close()

    # ── METRICS ───────────────────────────────────────────────────────────
    elapsed   = time.perf_counter() - t_start
    avg_size  = sum(page_sizes) / len(page_sizes) if page_sizes else 0
    avg_time  = sum(page_times) / len(page_times) if page_times else 0
    total_cats = len(categories)
    avg_items  = (sum(cat_totals.values()) / len(cat_totals)) if cat_totals else 500
    full_est_pages = int(avg_items / ITEMS_PER_PAGE * total_cats) + total_cats
    est_bytes  = full_est_pages * avg_size
    est_gb     = est_bytes / 1024**3

    # ── REPORT ────────────────────────────────────────────────────────────
    print()
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("  tonitrus.com — DISCOVERY REPORT")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print()
    print("CATALOGUE (from nav HTML)")
    print(f"  Total categories          : {total_cats:,}")
    for sec, cats in sorted(by_section.items()):
        print(f"    {sec:<22}: {len(cats):>4} categories")
    print()
    print(f"SAMPLED CATEGORIES          ({len(sample_cats)} of {total_cats})")
    for url, tot in cat_totals.items():
        name  = next((c["name"] for c in categories if c["url"] == url), url)
        pages = max(1, -(-tot // ITEMS_PER_PAGE))
        print(f"  {name:<40} {tot:>5} items  ({pages} pages)")
    print()
    print("PROXY")
    print(f"  Stack                     : Camoufox + Bright Data datacenter")
    print(f"  Auth method               : sticky token per category")
    print(f"  Health check              : {'PASSED' if proxy_ok else 'FAILED'}")
    print()
    print(f"SAMPLE  ({len(page_sizes)} pages fetched)")
    print(f"  Avg page size             : {avg_size/1024:.1f} KB")
    print(f"  Avg response time         : {avg_time:.2f}s")
    print(f"  Products parsed           : {len(all_products)}")
    print(f"  PDP-required products     : {len(pdp_candidates)}")
    print(f"  Total elapsed             : {elapsed:.1f}s")
    print()
    print("REQUEST BUDGET  (full run estimate)")
    print(f"  Total categories          : {total_cats:,}")
    print(f"  Est. PLP pages            : {full_est_pages:,}  (@ {ITEMS_PER_PAGE} items/page)")
    print(f"  Est. total bandwidth      : {est_gb:.2f} GB  ({est_bytes/1024**2:.0f} MB)")
    print(f"  Discovery consumed        : {total_bytes/1024:.1f} KB")
    print()

    if pdp_sample:
        print("SAMPLE PDP VARIANTS (first PDP-required product)")
        for v in pdp_sample:
            print(f"  condition  : {v['condition']}")
            print(f"  sku        : {v['sku']}")
            print(f"  price_gross: {v['price_gross']}")
            print(f"  stock      : {v['stock']}  oos={v['is_oos']}")
            print()

    print("SAMPLE PRODUCTS  (first 5)")
    for p in all_products[:5]:
        print(f"  handle       : {p['handle']}")
        print(f"  name         : {p['name']}")
        print(f"  brand        : {p['brand']}")
        print(f"  mpn          : {p['mpn']}")
        print(f"  ean          : {p['ean']}")
        print(f"  price        : {p['price_min']} – {p['price_max']}")
        print(f"  availability : {p['availability']}")
        print(f"  is_cto       : {p['is_cto']}")
        print(f"  pdp_required : {p['pdp_required']}")
        print()

    # ── save JSON ─────────────────────────────────────────────────────────
    ts       = time.strftime("%Y%m%d_%H%M%S")
    out_path = f"tonitrus_discovery_{ts}.json"
    payload  = dict(
        run_ts=ts,
        categories=categories,
        cat_totals=cat_totals,
        products=all_products,
        metrics=dict(
            proxy_ok=proxy_ok,
            total_categories=total_cats,
            sampled_categories=len(sample_cats),
            pages_fetched=len(page_sizes),
            products_parsed=len(all_products),
            avg_page_size_kb=round(avg_size/1024, 1),
            avg_response_time_s=round(avg_time, 2),
            discovery_bytes=total_bytes,
            est_full_pages=full_est_pages,
            est_bandwidth_gb=round(est_gb, 3),
        ),
    )
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    log.info("saved → %s  (%d products, %d categories)", out_path, len(all_products), total_cats)
    print(f"✓ Output saved → {out_path}")


log.info("discovery() coroutine defined — ready to run")

14:30:40  INFO      tonitrus  discovery() coroutine defined — ready to run


In [13]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# Works in standard .py script, in Jupyter (nest_asyncio), and in ipynb
asyncio.run(discovery())


14:30:42  DEBUG     tonitrus  proxy session  user=brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.3158742405355041
14:30:45  INFO      tonitrus  Proxy health-check …
14:30:45  DEBUG     tonitrus  GET  url=https://www.tonitrus.com/?lang=eng
14:30:51  DEBUG     tonitrus  OK   status=200  len=1537287  t=6.07s
14:30:51  INFO      tonitrus  Proxy health-check  PASS=True  t=6.07s  size=1537384 B
14:30:51  INFO      tonitrus  pass0  extracting categories from nav HTML …
14:30:52  DEBUG     tonitrus  pass0  section=Server
14:30:52  DEBUG     tonitrus  pass0  section=Storage
14:30:52  DEBUG     tonitrus  pass0  section=Desktop PC
14:30:53  INFO      tonitrus  pass0  extracted 734 unique categories
14:30:53  INFO      tonitrus  pass0  734 categories across 3 sections
14:30:53  DEBUG     tonitrus  proxy session  user=brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.8263215403852580
14:30:55  INFO      tonitrus  pass1  cat=HPE ProLiant G7 / Gen8  url=https://www.tonitrus.com/HPE


╔══════════════════════════════════════════════════════════════════╗
  tonitrus.com — DISCOVERY REPORT
╚══════════════════════════════════════════════════════════════════╝

CATALOGUE (from nav HTML)
  Total categories          : 734
    Desktop PC            :  156 categories
    Server                :  369 categories
    Storage               :  209 categories

SAMPLED CATEGORIES          (5 of 734)
  HPE ProLiant G7 / Gen8                       6 items  (1 pages)
  HP ProLiant DL20 Gen10                       8 items  (1 pages)
  HP ProLiant DL20 Gen10 Plus                  9 items  (1 pages)
  HP ProLiant DL20 Gen11                       4 items  (1 pages)
  HP ProLiant DL160 Gen10                      7 items  (1 pages)

PROXY
  Stack                     : Camoufox + Bright Data datacenter
  Auth method               : sticky token per category
  Health check              : PASSED

SAMPLE  (5 pages fetched)
  Avg page size             : 1417.2 KB
  Avg response time         : 7.2

In [30]:
"""
Tonitrus Category Discovery Script
===================================
Two-pass discovery:
  Pass 1 — Parse nav (ul.navbar-nav.nav-scrollbar-inner) for top-level cats + brand subcategories
  Pass 2 — Crawl each brand page to find leaf categories (div.sub-categories)
  Pass 3 — Crawl each leaf PLP (page 1, ?af=50) to get product count → derive pages, traffic

Per-leaf stats collected:
  - product_count   : total from "Items 1 - 50 of 669"
  - pages           : ceil(product_count / PER_PAGE)
  - plp_requests    : pages (one fetch per page)
  - pdp_requests    : product_count (one fetch per PDP)
  - total_requests  : plp_requests + pdp_requests

Outputs: discovery_report.json
"""

import asyncio
import json
import logging
import math
import random
import re
import time
from bs4 import BeautifulSoup
from camoufox.async_api import AsyncCamoufox

# ── Logging ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("discovery")

# ── Config ────────────────────────────────────────────────────────────────────

BASE_URL   = "https://www.tonitrus.com"
LANG       = "?lang=eng"
HOME_URL   = f"{BASE_URL}/{LANG}"
TIMEOUT_MS = 60_000
DELAY      = (1.5, 3.0)   # random sleep between requests
PER_PAGE   = 50            # force 50 items/page via ?af=50

PROXY_HOST = "brd.superproxy.io"
PROXY_PORT = 33335
PROXY_USER = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS = "ymg5cg3a1z33"

# Top-level cats to skip (not product pages)
SKIP_CATS = {"Services", "Unternehmen", "Leistungen", ""}

def make_proxy(token: str | None = None) -> dict:
    if token is None:
        token = f"{random.random():.16f}"
    user = f"{PROXY_USER}-session-{token}"
    return {
        "server":   f"http://{PROXY_HOST}:{PROXY_PORT}",
        "username": user,
        "password": PROXY_PASS,
    }

def abs_url(href: str) -> str:
    """Make relative URLs absolute."""
    if not href:
        return ""
    if href.startswith("http"):
        return href
    return BASE_URL + href

def add_lang(url: str) -> str:
    if not url:
        return url
    if "?" in url:
        return url + "&lang=eng"
    return url + "?lang=eng"

def parse_product_count(html: str) -> int | None:
    """
    Extract total product count from PLP.
    Looks for: div.productlist-item-info text like "Items 1 - 50 of 669"
    """
    soup = BeautifulSoup(html, "html.parser")
    info = soup.select_one("div.productlist-item-info")
    if info:
        text = info.get_text(strip=True)
        m = re.search(r'of\s+([\d,\.]+)', text, re.I)
        if m:
            return int(m.group(1).replace(",", "").replace(".", ""))
    # Fallback: count product cards directly (only works for single-page cats)
    cards = soup.select("a.text-clamp-2[href]")
    return len(cards) if cards else None

def plp_url(base: str, page: int = 1) -> str:
    """
    Construct PLP URL for a given page.
    Page 1:  {base}?af=50&lang=eng
    Page N:  {base}_sN?af=50&lang=eng
    Strips any existing _sN suffix before rebuilding.
    """
    # Remove existing _sN suffix
    clean = re.sub(r'_s\d+$', '', base.rstrip('/'))
    # Remove existing query string
    clean = clean.split('?')[0]
    suffix = f"_s{page}" if page > 1 else ""
    return f"{clean}{suffix}?af={PER_PAGE}&lang=eng"

# ── Pass 1: Parse Nav ─────────────────────────────────────────────────────────

def parse_nav(html: str) -> dict:
    """
    Returns structure:
    {
      "Konfigurator": {
        "url": "...",
        "type": "configurator",
        "leaves": [{"name": ..., "url": ...}, ...]
      },
      "Server": {
        "url": "...",
        "type": "category",
        "brands": [{"name": ..., "url": ...}, ...]
      },
      ...
    }
    """
    soup = BeautifulSoup(html, "html.parser")

    # Correct selector: ul with both navbar-nav AND nav-scrollbar-inner classes
    nav = soup.find("ul", class_=lambda c: c and "navbar-nav" in c and "nav-scrollbar-inner" in c)
    if not nav:
        log.error("Nav not found! Check selector.")
        return {}

    result = {}
    items = nav.find_all("li", class_="nav-item", recursive=False)
    log.info("Found %d top-level nav items", len(items))

    for item in items:
        a = item.find("a", class_="nav-link")
        if not a:
            continue

        name = a.get_text(strip=True)
        href = abs_url(a.get("href", ""))
        cat_id = a.get("data-category-id", "")

        if not name or name in SKIP_CATS:
            continue

        dropdown = item.find("div", class_="dropdown-menu")
        if not dropdown:
            continue

        # Configurator: has direct leaf links in nav (no brand tier)
        if name == "Konfigurator" or (cat_id and cat_id == "4715"):
            leaves = []
            for brand_a in dropdown.find_all("a", class_="list-group-item"):
                lname = brand_a.get_text(strip=True)
                lhref = brand_a.get("href", "") or ""
                onclick = brand_a.get("onclick", "")
                if "window.location=" in onclick:
                    lhref = onclick.split("'")[1]
                if lname and lhref:
                    leaves.append({"name": lname, "url": add_lang(abs_url(lhref))})
            result["Konfigurator"] = {
                "url": href,
                "type": "configurator",
                "leaves": leaves,
            }
            log.info("Konfigurator: %d direct leaves", len(leaves))

        else:
            # Regular category: list-group-items are brand pages
            brands = []
            for brand_a in dropdown.find_all("a", class_="list-group-item"):
                bname = brand_a.get_text(strip=True)
                bhref = brand_a.get("href", "") or ""
                onclick = brand_a.get("onclick", "")
                if "window.location=" in onclick:
                    bhref = onclick.split("'")[1]
                if bname and bhref:
                    brands.append({"name": bname, "url": add_lang(abs_url(bhref))})
            result[name] = {
                "url": href,
                "type": "category",
                "brands": brands,
            }
            log.info("Category %s: %d brands", name, len(brands))

    return result

# ── Pass 2: Crawl Brand Pages for Leaf Cats ───────────────────────────────────

def parse_brand_page(html: str) -> list[dict]:
    """Extract leaf categories from a brand page (div.sub-categories)."""
    soup = BeautifulSoup(html, "html.parser")
    leaves = []
    for div in soup.find_all("div", class_="sub-categories"):
        caption = div.find("div", class_="caption")
        if caption:
            a = caption.find("a")
            if a:
                lname = a.get_text(strip=True)
                lhref = abs_url(a.get("href", ""))
                if lname and lhref:
                    leaves.append({"name": lname, "url": add_lang(lhref)})
    return leaves

# ── Browser Fetch ─────────────────────────────────────────────────────────────

async def fetch_page(browser, url: str) -> str | None:
    """Fetch a page and return its HTML. Returns None on failure."""
    page = None
    try:
        page = await browser.new_page()
        await page.goto(url, timeout=TIMEOUT_MS, wait_until="domcontentloaded")
        # Wait for nav to be present
        try:
            await page.wait_for_selector("ul.navbar-nav", timeout=15_000)
        except Exception:
            pass
        html = await page.content()
        return html
    except Exception as e:
        log.warning("Failed to fetch %s: %s", url, e)
        return None
    finally:
        if page:
            await page.close()

# ── Main ──────────────────────────────────────────────────────────────────────

async def main():
    report = {
        "scraped_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "base_url": BASE_URL,
        "categories": {}
    }

    async with AsyncCamoufox(headless=True, proxy=make_proxy()) as browser:

        # ── Pass 1: Fetch homepage and parse nav ──────────────────────────────
        log.info("Pass 1: Fetching homepage...")
        home_html = await fetch_page(browser, HOME_URL)
        if not home_html:
            log.error("Failed to fetch homepage. Aborting.")
            return

        nav_structure = parse_nav(home_html)
        log.info("Nav parsed: %d top-level categories", len(nav_structure))

        # ── Pass 2: Crawl brand pages for leaf categories ─────────────────────
        for cat_name, cat_data in nav_structure.items():
            report["categories"][cat_name] = {
                "url": cat_data["url"],
                "type": cat_data["type"],
                "brands": {}
            }

            if cat_data["type"] == "configurator":
                # Configurator leaves come directly from nav
                report["categories"][cat_name]["leaves"] = cat_data["leaves"]
                log.info("[Konfigurator] %d leaves (from nav)", len(cat_data["leaves"]))
                continue

            # For regular categories, crawl each brand page
            for brand in cat_data.get("brands", []):
                bname = brand["name"]
                burl  = brand["url"]
                log.info("  Crawling brand: %s → %s", bname, burl)

                await asyncio.sleep(random.uniform(*DELAY))
                brand_html = await fetch_page(browser, burl)

                leaves = []
                if brand_html:
                    leaves = parse_brand_page(brand_html)
                    log.info("    → %d leaf categories", len(leaves))
                else:
                    log.warning("    → failed to fetch brand page")

                # ── Pass 3: crawl each leaf PLP for product count ────────────
                enriched_leaves = []
                for leaf in leaves:
                    lname = leaf["name"]
                    lbase = leaf["url"].split("?")[0]  # strip any existing params
                    lurl  = plp_url(lbase, page=1)

                    log.info("      Leaf PLP: %s → %s", lname, lurl)
                    await asyncio.sleep(random.uniform(*DELAY))
                    leaf_html = await fetch_page(browser, lurl)

                    product_count = None
                    if leaf_html:
                        product_count = parse_product_count(leaf_html)

                    pages         = math.ceil(product_count / PER_PAGE) if product_count else None
                    plp_requests  = pages if pages else None
                    pdp_requests  = product_count if product_count else None
                    total_requests = (plp_requests + pdp_requests) if (plp_requests and pdp_requests) else None

                    enriched_leaves.append({
                        "name":           lname,
                        "url":            lbase,
                        "product_count":  product_count,
                        "pages":          pages,
                        "plp_requests":   plp_requests,
                        "pdp_requests":   pdp_requests,
                        "total_requests": total_requests,
                    })
                    log.info("        → %s products, %s pages, %s total requests",
                             product_count, pages, total_requests)

                report["categories"][cat_name]["brands"][bname] = {
                    "url":    burl,
                    "leaves": enriched_leaves,
                }
    # ── Summary ───────────────────────────────────────────────────────────────
    total_leaves        = 0
    total_products      = 0
    total_plp_requests  = 0
    total_pdp_requests  = 0
    total_requests      = 0

    for cat_name, cat_data in report["categories"].items():
        if cat_data["type"] == "configurator":
            # Konfigurator leaves don't have product counts yet (CTO pages are different)
            total_leaves += len(cat_data.get("leaves", []))
        else:
            for bname, bdata in cat_data.get("brands", {}).items():
                for leaf in bdata.get("leaves", []):
                    total_leaves       += 1
                    pc  = leaf.get("product_count") or 0
                    plp = leaf.get("plp_requests")  or 0
                    pdp = leaf.get("pdp_requests")  or 0
                    tr  = leaf.get("total_requests") or 0
                    total_products     += pc
                    total_plp_requests += plp
                    total_pdp_requests += pdp
                    total_requests     += tr

    report["summary"] = {
        "top_level_categories": len(report["categories"]),
        "total_leaf_categories": total_leaves,
        "total_products":        total_products,
        "total_plp_requests":    total_plp_requests,
        "total_pdp_requests":    total_pdp_requests,
        "total_requests":        total_requests,
        "items_per_page":        PER_PAGE,
    }

    out_path = "discovery_report.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    log.info("=" * 60)
    log.info("Discovery complete!")
    log.info("  Top-level categories : %d", report["summary"]["top_level_categories"])
    log.info("  Total leaf categories: %d", total_leaves)
    log.info("  Total products       : %d", total_products)
    log.info("  Total PLP requests   : %d", total_plp_requests)
    log.info("  Total PDP requests   : %d", total_pdp_requests)
    log.info("  Grand total requests : %d", total_requests)
    log.info("  Report saved to      : %s", out_path)
    log.info("=" * 60)

if __name__ == "__main__":
    asyncio.run(main())

17:01:25  INFO      discovery  Pass 1: Fetching homepage...
17:01:31  INFO      discovery  Found 9 top-level nav items
17:01:31  INFO      discovery  Konfigurator: 12 direct leaves
17:01:31  INFO      discovery  Category Server: 7 brands
17:01:31  INFO      discovery  Category Storage: 9 brands
17:01:31  INFO      discovery  Category Networking: 13 brands
17:01:31  INFO      discovery  Category Desktop PC: 7 brands
17:01:31  INFO      discovery  Category Notebook: 4 brands
17:01:31  INFO      discovery  Category Company: 0 brands
17:01:31  INFO      discovery  Nav parsed: 7 top-level categories
17:01:31  INFO      discovery  [Konfigurator] 12 leaves (from nav)
17:01:31  INFO      discovery    Crawling brand: HP → https://www.tonitrus.com/HP_30?lang=eng
17:01:41  INFO      discovery      → 4 leaf categories
17:01:41  INFO      discovery        Leaf PLP: Rack Server → https://www.tonitrus.com/Rack-Server_15?af=50&lang=eng
17:01:52  INFO      discovery          → 717 products, 15 pages, 7

In [18]:
import asyncio, re
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from camoufox.async_api import AsyncCamoufox

PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 33335
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

PLP_URL      = "https://www.tonitrus.com/Switch_21?lang=eng&af=10"
CATEGORY_URL = "https://www.tonitrus.com/Switch_21"

def make_proxy():
    return {
        "server":   f"http://{PROXY_HOST}:{PROXY_PORT}",
        "username": PROXY_USER,
        "password": PROXY_PASS,
    }

def parse_plp_cards(html: str) -> list[dict]:
    soup  = BeautifulSoup(html, "lxml")
    cards = soup.select("div[itemprop='itemListElement']")
    products = []
    for card in cards:
        a_tags = card.select("a[href]")
        if not a_tags:
            continue
        url      = a_tags[0]["href"]
        handle   = urlparse(url).path.lstrip("/")
        raw_name = a_tags[0].get_text(strip=True)

        # brand = first segment, mpn = second segment
        parts = [p.strip() for p in raw_name.split(" - ", 2)]
        brand = parts[0] if len(parts) > 0 else ""
        mpn   = parts[1] if len(parts) > 1 else ""

        products.append(dict(
            url=url,
            handle=handle,
            full_name=raw_name,   # entire raw string, cleaned up on PDP
            brand=brand,
            mpn=mpn,
        ))
    return products


def parse_pdp(html: str, input_url: str) -> dict:
    soup = BeautifulSoup(html, "lxml")

    # ── product name — full title from <h1> ──────────────────────────────
    h1 = soup.select_one("h1.product-title, h1[itemprop='name'], h1")
    product_name = h1.get_text(strip=True) if h1 else ""

    # ── base SKU ─────────────────────────────────────────────────────────
    base_sku = ""
    sku_tag  = soup.select_one("span[itemprop='sku']")
    if sku_tag:
        base_sku = sku_tag.get_text(strip=True)

    # ── breadcrumb — ordered by itemprop position, skip Homepage + last ──
    crumbs = []
    for li in soup.select("li.breadcrumb-item[itemprop='itemListElement']"):
        pos_tag  = li.select_one("meta[itemprop='position']")
        name_tag = li.select_one("span[itemprop='name']")
        href_tag = li.select_one("a[itemprop='url']")
        if not name_tag:
            continue
        pos  = int(pos_tag["content"]) if pos_tag else 99
        name = name_tag.get_text(strip=True)
        href = href_tag["href"] if href_tag else ""
        crumbs.append((pos, name, href))

    crumbs.sort(key=lambda x: x[0])
    # drop position 1 (Homepage) and last (product itself)
    crumbs = crumbs[1:-1]
    subcategory = " > ".join(c[1] for c in crumbs)  # e.g. "Networking > Cisco > Switch > Cisco Catalyst 1000 Switch"

    # ── model — from breadcrumb second-to-last or mpn ────────────────────
    model = crumbs[-1][1] if crumbs else ""

    # ── prices ───────────────────────────────────────────────────────────
    prices = {}
    for row in soup.select("div.lpxBorderVar"):
        radio    = row.select_one("input[type='radio']")
        cond_raw = radio.get("aria-label", "").strip().upper() if radio else ""
        if "NEW" in cond_raw:
            cond = "NEW"
        elif "REFURBISHED" in cond_raw:
            cond = "REFURBISHED"
        else:
            continue

        # variant SKU
        small_tag = row.select_one("small")
        var_sku   = small_tag.get_text(strip=True).replace("SKU:", "").strip() if small_tag else ""

        # OOS
        is_oos = bool(row.select_one("span.badge-not-available, label[data-stock='out-of-stock']"))

        # price gross
        pg_tag      = row.select_one("span.lpxPDetailsPreis")
        price_gross = pg_tag.get_text(strip=True) if pg_tag else None

        # price net
        price_net = None
        for s in row.select("small"):
            t = s.get_text(strip=True)
            if t.startswith("Net:"):
                price_net = t.replace("Net:", "").strip()
                break

        # stock
        stock_tag = row.select_one("span.status")
        stock     = int(stock_tag.get_text(strip=True)) if stock_tag and stock_tag.get_text(strip=True).isdigit() else 0

        status = "OOS" if is_oos or not price_gross else "InStock"

        prices[cond] = dict(
            sku=var_sku,
            price_gross=price_gross,
            price_net=price_net,
            stock=stock,
            status=status,
        )

    return dict(
        product_name=product_name,
        base_sku=base_sku,
        model=model,
        subcategory=subcategory,
        prices=prices,
        input_url=input_url,
    )


async def test():
    import json, time
    results = []

    async with AsyncCamoufox(headless=True, geoip=True, proxy=make_proxy()) as browser:

        # ── PLP ───────────────────────────────────────────────────────────
        page = await browser.new_page()
        print(f"PLP  {PLP_URL}")
        await page.goto(PLP_URL, wait_until="domcontentloaded", timeout=90000)
        products = parse_plp_cards(await page.content())
        await page.close()
        print(f"found {len(products)} products\n")

        # ── PDP for all products ──────────────────────────────────────────
        for prod in products:
            pdp_url = prod["url"] + "?lang=eng"
            print(f"PDP  {pdp_url}")
            t0   = time.perf_counter()
            page = await browser.new_page()
            await page.goto(pdp_url, wait_until="domcontentloaded", timeout=90000)
            pdp  = parse_pdp(await page.content(), input_url=CATEGORY_URL)
            await page.close()
            print(f"     {time.perf_counter()-t0:.1f}s")

            record = dict(
                url          = prod["url"],
                product_name = pdp["product_name"],
                brand        = prod["brand"],
                model        = pdp["model"],
                mpn          = prod["mpn"],
                barcode      = pdp["base_sku"],
                subcategory  = pdp["subcategory"],
                prices       = pdp["prices"],
                input_url    = pdp["input_url"],
            )
            results.append(record)

            # pretty print
            print(f"  product_name : {record['product_name']}")
            print(f"  brand        : {record['brand']}")
            print(f"  model        : {record['model']}")
            print(f"  mpn          : {record['mpn']}")
            print(f"  barcode      : {record['barcode']}")
            print(f"  subcategory  : {record['subcategory']}")
            print(f"  input_url    : {record['input_url']}")
            for cond, info in record["prices"].items():
                print(f"  {cond:<12} sku={info['sku']}  gross={info['price_gross']}  "
                      f"net={info['price_net']}  stock={info['stock']}  status={info['status']}")
            print()

    # ── save JSON ─────────────────────────────────────────────────────────
    ts       = time.strftime("%Y%m%d_%H%M%S")
    out_path = f"tonitrus_sample_{ts}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✓ saved {len(results)} products → {out_path}")


asyncio.run(test())

PLP  https://www.tonitrus.com/Switch_21?lang=eng&af=10
found 10 products

PDP  https://www.tonitrus.com/C1000-16FP-2G-L_16?lang=eng
     7.7s
  product_name : Cisco - C1000-16FP-2G-L - Cisco - C1000-16FP-2G-L - Catalyst 1000 16port GE, Full POE, 2x1G SFP
  brand        : Cisco
  model        : Cisco Catalyst 1000 Switch
  mpn          : C1000-16FP-2G-L
  barcode      : 10237444
  subcategory  : Networking > Cisco > Switch > Cisco Catalyst 1000 Switch
  input_url    : https://www.tonitrus.com/Switch_21
  NEW          sku=10237444-003  gross=None  net=None  stock=0  status=OOS
  REFURBISHED  sku=10237444-014  gross=1.078,50 €  net=906,30 €  stock=1  status=InStock

PDP  https://www.tonitrus.com/C1000-16P-2G-L_17?lang=eng
     5.1s
  product_name : Cisco - C1000-16P-2G-L - Cisco - C1000-16P-2G-L - Catalyst 1000 16port GE, POE, 2x1G SFP
  brand        : Cisco
  model        : Cisco Catalyst 1000 Switch
  mpn          : C1000-16P-2G-L
  barcode      : 10237339
  subcategory  : Networking > 

In [24]:
CTO_URL      = "https://www.tonitrus.com/Cisco-C240-M5-CTO-Server_1?lang=eng"
CATEGORY_URL = "https://www.tonitrus.com/Server_1"

def parse_price_float(s):
    if not s: return None
    return float(s.replace("€","").replace(".","").replace(",",".").strip())

def format_price_eur(val):
    if val is None: return None
    # format as German: 1.234,56 €
    formatted = f"{val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"{formatted} €"

async def test_cto():
    async with AsyncCamoufox(headless=True, geoip=True, proxy=make_proxy()) as browser:
        page = await browser.new_page()
        print(f"[CTO] loading {CTO_URL}")
        await page.goto(CTO_URL, wait_until="domcontentloaded", timeout=90_000)

        # wait for JS to render configurator
        await page.wait_for_selector("label.variation.swatches-text", timeout=30_000)

        # click first chassis to populate price + config groups
        await page.click("label.variation.swatches-text")
        await page.wait_for_selector(".price.h1", timeout=15_000)
        await page.wait_for_selector("div.cfg-group", timeout=15_000)

        html = await page.content()
        soup = BeautifulSoup(html, "lxml")

        # ── debug ────────────────────────────────────────────────────────
        print("[DEBUG] title       :", soup.title.get_text() if soup.title else "N/A")
        print("[DEBUG] cfg-groups  :", len(soup.select("div.cfg-group")))
        print("[DEBUG] chassis lbls:", len(soup.select("label.variation.swatches-text")))
        print("[DEBUG] price.h1    :", soup.select_one(".price.h1"))

        # ── product name ─────────────────────────────────────────────────
        h1 = soup.select_one("h1.product-title, h1[itemprop='name'], h1")
        product_name = h1.get_text(strip=True) if h1 else ""

        # ── brand / mpn (skip repeated segments) ─────────────────────────
        parts = [p.strip() for p in product_name.split(" - ")]
        brand = parts[0]
        mpn   = brand  # fallback
        for i in range(1, len(parts)):
            if parts[i] != brand:
                mpn = parts[i]
                break

        # ── barcode / SKU ─────────────────────────────────────────────────
        sku_tag = soup.select_one("span[itemprop='sku']")
        barcode = sku_tag.get_text(strip=True) if sku_tag else ""

        # ── subcategory ───────────────────────────────────────────────────
        crumbs = []
        for li in soup.select("li.breadcrumb-item[itemprop='itemListElement']"):
            pos_tag  = li.select_one("meta[itemprop='position']")
            name_tag = li.select_one("span[itemprop='name']")
            if not name_tag: continue
            crumbs.append((int(pos_tag["content"]) if pos_tag else 99, name_tag.get_text(strip=True)))
        crumbs.sort(key=lambda x: x[0])
        subcategory = " > ".join(c[1] for c in crumbs[1:-1])

        # ── chassis list ──────────────────────────────────────────────────
        chassis_list = []
        for label in soup.select("label.variation.swatches-text"):
            chassis_list.append({
                "name":            label.get("data-original", "").strip(),
                "base_price_gross": None,
            })

        # ── per-chassis price (wait for text change) ──────────────────────
        labels = await page.query_selector_all("label.variation.swatches-text")
        for i, chassis in enumerate(chassis_list):
            prev_price = chassis_list[i-1]["base_price_gross"] if i > 0 else ""
            await labels[i].click()
            await page.wait_for_function(
                f"""() => {{
                    const el = document.querySelector('.price.h1');
                    return el && el.innerText.trim() !== {repr(prev_price or '')};
                }}""",
                timeout=15_000
            )
            price_el = await page.query_selector(".price.h1")
            chassis["base_price_gross"] = (await price_el.inner_text()).strip() if price_el else None
            print(f"[CHASSIS] {chassis['name']} → {chassis['base_price_gross']}")

        # re-grab html after last chassis click for config groups
        html = await page.content()
        soup = BeautifulSoup(html, "lxml")

        # ── prices ────────────────────────────────────────────────────────
        max_meta      = soup.select_one("meta[itemprop='maxPrice']")
        chassis_vals  = [parse_price_float(c["base_price_gross"]) for c in chassis_list if c["base_price_gross"]]
        min_price_val = min(chassis_vals) if chassis_vals else None

        prices = {
            "min_price_gross": format_price_eur(min_price_val),
            "max_price_gross": f"{float(max_meta['content']):,.2f} €".replace(",","X").replace(".",",").replace("X",".") if max_meta else None,
            "tax":             "19% VAT included, no net available",
        }

        # ── config options ────────────────────────────────────────────────
        config_options = {}
        for grp in soup.select("div.cfg-group"):
            name_el  = grp.select_one(".hr-sect.h3")
            grp_name = name_el.get_text(strip=True) if name_el else f"group_{grp.get('data-id', '')}"
            options  = []
            for item_div in grp.select("div.config-item"):
                desc_el   = item_div.select_one(".cfg-item-description-description")
                price_col = item_div.select_one(".cfg-item-qty.text-right")
                options.append({
                    "name":        desc_el.get_text(strip=True) if desc_el else "",
                    "price_delta": price_col.get_text(strip=True) if price_col else None,
                    "available":   "disabled" not in item_div.get("class", []),
                })
            config_options[grp_name] = options

        # ── assemble ──────────────────────────────────────────────────────
        record = {
            "url":            CTO_URL,
            "product_name":   product_name,
            "brand":          brand,
            "mpn":            mpn,
            "barcode":        barcode,
            "subcategory":    subcategory,
            "input_url":      CATEGORY_URL,
            "product_type":   "CTO",
            "prices":         prices,
            "chassis":        chassis_list,
            "config_options": config_options,
        }

        import json, pathlib
        print("\n[RESULT]")
        print(json.dumps(record, indent=2, ensure_ascii=False))

        # ── save to disk ──────────────────────────────────────────────────
        out_path = pathlib.Path("CTO_Test.json")
        out_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"\n[SAVED] → {out_path.resolve()}")

        await page.close()

await test_cto()

[CTO] loading https://www.tonitrus.com/Cisco-C240-M5-CTO-Server_1?lang=eng
[DEBUG] title       : Cisco - Cisco C240 M5 CTO Server - Cisco C240 M5 CTO Server
[DEBUG] cfg-groups  : 12
[DEBUG] chassis lbls: 2
[DEBUG] price.h1    : <div class="price h1">
<span itemprop="priceSpecification" itemscope="" itemtype="https://schema.org/UnitPriceSpecification">
                                                    494,50 €                                                                        </span>
</div>
[CHASSIS] Cisco UCS C240 M5 - 24x SFF Server → 494,50 €
[CHASSIS] Cisco UCS C240 M5 - 8x SFF Server → 238,50 €

[RESULT]
{
  "url": "https://www.tonitrus.com/Cisco-C240-M5-CTO-Server_1?lang=eng",
  "product_name": "Cisco  - Cisco - Cisco UCS C240 M5 - 24x SFF Server - Cisco UCS C240 M5 - 24x SFF Server",
  "brand": "Cisco",
  "mpn": "Cisco UCS C240 M5",
  "barcode": "TCO_00109-014",
  "subcategory": "Server > Cisco > Rack > Cisco UCS C240 M5",
  "input_url": "https://www.tonitrus.com/Server_1",

In [27]:
import asyncio, json, pathlib, time
from bs4 import BeautifulSoup
from camoufox.async_api import AsyncCamoufox

PLP_URL      = "https://www.tonitrus.com/Rack_26?lang=eng"
CATEGORY_URL = "https://www.tonitrus.com/Rack_26"
OUT_FILE     = pathlib.Path("Rack_26_test.json")
MAX_RETRIES  = 3
BACKOFF_BASE = 5  # seconds — doubles each retry: 5, 10, 20

def make_proxy():
    return {
        "server":   "http://brd.superproxy.io:33335",
        "username": "brd-customer-hl_14d32bce-zone-datacenter_proxy1",
        "password": "ymg5cg3a1z33",
    }

def parse_price_float(s):
    if not s: return None
    return float(s.replace("€","").replace(".","").replace(",",".").strip())

def format_price_eur(val):
    if val is None: return None
    return f"{val:,.2f}".replace(",","X").replace(".",",").replace("X",".") + " €"

# ── PLP ───────────────────────────────────────────────────────────────────────

def parse_plp_cards(html):
    soup     = BeautifulSoup(html, "lxml")
    products = []
    for card in soup.select("div[itemprop='itemListElement']"):
        a = card.select_one("a[href]")
        if not a: continue
        raw_name = a.get_text(strip=True)
        parts    = [p.strip() for p in raw_name.split(" - ", 2)]
        products.append({
            "url":    a["href"],
            "brand":  parts[0] if parts else "",
            "mpn":    parts[1] if len(parts) > 1 else "",
            "is_cto": "CTO" in raw_name.upper(),
        })
    return products

# ── PDP (standard) ────────────────────────────────────────────────────────────

def parse_pdp(html, input_url):
    soup         = BeautifulSoup(html, "lxml")
    h1           = soup.select_one("h1.product-title, h1[itemprop='name'], h1")
    product_name = h1.get_text(strip=True) if h1 else ""

    parts = [p.strip() for p in product_name.split(" - ")]
    brand = parts[0]
    mpn   = brand
    for i in range(1, len(parts)):
        if parts[i] != brand:
            mpn = parts[i]
            break

    sku_tag = soup.select_one("span[itemprop='sku']")
    barcode = sku_tag.get_text(strip=True) if sku_tag else ""

    crumbs = []
    for li in soup.select("li.breadcrumb-item[itemprop='itemListElement']"):
        pos_tag  = li.select_one("meta[itemprop='position']")
        name_tag = li.select_one("span[itemprop='name']")
        if not name_tag: continue
        crumbs.append((int(pos_tag["content"]) if pos_tag else 99, name_tag.get_text(strip=True)))
    crumbs.sort(key=lambda x: x[0])
    subcategory = " > ".join(c[1] for c in crumbs[1:-1])
    model       = crumbs[-1][1] if crumbs else ""

    prices = {}
    for row in soup.select("div.lpxBorderVar"):
        radio    = row.select_one("input[type='radio']")
        cond_raw = radio.get("aria-label","").strip().upper() if radio else ""
        if "NEW" in cond_raw:            cond = "NEW"
        elif "REFURBISHED" in cond_raw:  cond = "REFURBISHED"
        else: continue

        small_tag   = row.select_one("small")
        var_sku     = small_tag.get_text(strip=True).replace("SKU:","").strip() if small_tag else ""
        is_oos      = bool(row.select_one("span.badge-not-available, label[data-stock='out-of-stock']"))
        pg_tag      = row.select_one("span.lpxPDetailsPreis")
        price_gross = pg_tag.get_text(strip=True) if pg_tag else None
        price_net   = None
        for s in row.select("small"):
            t = s.get_text(strip=True)
            if t.startswith("Net:"):
                price_net = t.replace("Net:","").strip()
                break
        stock_tag = row.select_one("span.status")
        stock     = int(stock_tag.get_text(strip=True)) if stock_tag and stock_tag.get_text(strip=True).isdigit() else 0
        prices[cond] = {
            "sku":         var_sku,
            "price_gross": price_gross,
            "price_net":   price_net,
            "stock":       stock,
            "status":      "OOS" if is_oos or not price_gross else "InStock",
        }

    return {
        "product_name": product_name,
        "brand":        brand,
        "model":        model,
        "mpn":          mpn,
        "barcode":      barcode,
        "subcategory":  subcategory,
        "input_url":    input_url,
        "product_type": "STANDARD",
        "prices":       prices,
    }

# ── PDP (CTO) ─────────────────────────────────────────────────────────────────
async def parse_cto(page, url, input_url):
    await page.goto(url, wait_until="domcontentloaded", timeout=90_000)

    # try to wait for chassis swatches — if OOS they may never appear
    try:
        await page.wait_for_selector("label.variation.swatches-text", timeout=15_000)
        has_chassis = True
    except:
        has_chassis = False
        print("    [OOS-CTO] no chassis swatches found, falling back to basic parse")

    html = await page.content()
    soup = BeautifulSoup(html, "lxml")

    # ── shared fields (always present) ───────────────────────────────────
    h1           = soup.select_one("h1.product-title, h1[itemprop='name'], h1")
    product_name = h1.get_text(strip=True) if h1 else ""

    parts = [p.strip() for p in product_name.split(" - ")]
    brand = parts[0]
    mpn   = brand
    for i in range(1, len(parts)):
        if parts[i] != brand:
            mpn = parts[i]
            break

    sku_tag = soup.select_one("span[itemprop='sku']")
    barcode = sku_tag.get_text(strip=True) if sku_tag else ""

    crumbs = []
    for li in soup.select("li.breadcrumb-item[itemprop='itemListElement']"):
        pos_tag  = li.select_one("meta[itemprop='position']")
        name_tag = li.select_one("span[itemprop='name']")
        if not name_tag: continue
        crumbs.append((int(pos_tag["content"]) if pos_tag else 99, name_tag.get_text(strip=True)))
    crumbs.sort(key=lambda x: x[0])
    subcategory = " > ".join(c[1] for c in crumbs[1:-1])
    model       = crumbs[-1][1] if crumbs else ""

    # ── OOS fallback — return skeleton with no prices/chassis/options ────
    if not has_chassis:
        return {
            "product_name":   product_name,
            "brand":          brand,
            "model":          model,
            "mpn":            mpn,
            "barcode":        barcode,
            "subcategory":    subcategory,
            "input_url":      input_url,
            "product_type":   "CTO",
            "status":         "OOS",
            "prices":         {},
            "chassis":        [],
            "config_options": {},
        }

    # ── normal CTO flow ───────────────────────────────────────────────────
    await page.click("label.variation.swatches-text")
    await page.wait_for_selector(".price.h1", timeout=15_000)
    await page.wait_for_selector("div.cfg-group", timeout=15_000)

    html = await page.content()
    soup = BeautifulSoup(html, "lxml")

    chassis_list = []
    for label in soup.select("label.variation.swatches-text"):
        chassis_list.append({"name": label.get("data-original","").strip(), "base_price_gross": None})

    labels = await page.query_selector_all("label.variation.swatches-text")
    for i, chassis in enumerate(chassis_list):
        prev_price = chassis_list[i-1]["base_price_gross"] if i > 0 else ""
        await labels[i].click()
        await page.wait_for_function(
            f"""() => {{
                const el = document.querySelector('.price.h1');
                return el && el.innerText.trim() !== {repr(prev_price or '')};
            }}""",
            timeout=15_000
        )
        price_el = await page.query_selector(".price.h1")
        chassis["base_price_gross"] = (await price_el.inner_text()).strip() if price_el else None
        print(f"    [CHASSIS] {chassis['name']} → {chassis['base_price_gross']}")

    html         = await page.content()
    soup         = BeautifulSoup(html, "lxml")
    max_meta     = soup.select_one("meta[itemprop='maxPrice']")
    chassis_vals = [parse_price_float(c["base_price_gross"]) for c in chassis_list if c["base_price_gross"]]
    min_val      = min(chassis_vals) if chassis_vals else None

    config_options = {}
    for grp in soup.select("div.cfg-group"):
        name_el  = grp.select_one(".hr-sect.h3")
        grp_name = name_el.get_text(strip=True) if name_el else f"group_{grp.get('data-id','')}"
        options  = []
        for item_div in grp.select("div.config-item"):
            desc_el   = item_div.select_one(".cfg-item-description-description")
            price_col = item_div.select_one(".cfg-item-qty.text-right")
            options.append({
                "name":        desc_el.get_text(strip=True) if desc_el else "",
                "price_delta": price_col.get_text(strip=True) if price_col else None,
                "available":   "disabled" not in item_div.get("class", []),
            })
        config_options[grp_name] = options

    return {
        "product_name":   product_name,
        "brand":          brand,
        "model":          model,
        "mpn":            mpn,
        "barcode":        barcode,
        "subcategory":    subcategory,
        "input_url":      input_url,
        "product_type":   "CTO",
        "status":         "InStock",
        "prices": {
            "min_price_gross": format_price_eur(min_val),
            "max_price_gross": format_price_eur(float(max_meta["content"])) if max_meta else None,
            "tax":             "19% VAT included, no net available",
        },
        "chassis":        chassis_list,
        "config_options": config_options,
    }

# ── RETRY WRAPPER ─────────────────────────────────────────────────────────────

async def scrape_with_retry(browser, prod, input_url):
    pdp_url = prod["url"] + "?lang=eng"
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            page = await browser.new_page()
            if prod["is_cto"]:
                print(f"  [CTO] {pdp_url}  (attempt {attempt})")
                data = await parse_cto(page, pdp_url, input_url)
            else:
                print(f"  [PDP] {pdp_url}  (attempt {attempt})")
                await page.goto(pdp_url, wait_until="domcontentloaded", timeout=90_000)
                data = parse_pdp(await page.content(), input_url)
            await page.close()
            return {"url": prod["url"], **data}

        except Exception as e:
            last_err = e
            await page.close()
            wait = BACKOFF_BASE * (2 ** (attempt - 1))  # 5, 10, 20
            print(f"  [RETRY {attempt}/{MAX_RETRIES}] {e} — waiting {wait}s")
            await asyncio.sleep(wait)

    print(f"  [FAILED] {pdp_url} — all retries exhausted")
    return {"url": prod["url"], "error": str(last_err), "product_type": "CTO" if prod["is_cto"] else "STANDARD"}

# ── MAIN ──────────────────────────────────────────────────────────────────────

async def test_rack():
    results = []

    async with AsyncCamoufox(headless=True, geoip=True, proxy=make_proxy()) as browser:
        page = await browser.new_page()
        print(f"[PLP] {PLP_URL}")
        await page.goto(PLP_URL, wait_until="domcontentloaded", timeout=90_000)
        products = parse_plp_cards(await page.content())
        await page.close()
        print(f"[PLP] found {len(products)} products ({sum(p['is_cto'] for p in products)} CTO)\n")

        for prod in products:
            t0     = time.perf_counter()
            record = await scrape_with_retry(browser, prod, CATEGORY_URL)
            results.append(record)

            if "error" not in record:
                print(f"  name : {record['product_name']}")
                print(f"  brand: {record['brand']}  mpn: {record['mpn']}  barcode: {record['barcode']}")
                if record["product_type"] == "CTO":
                    for c in record["chassis"]:
                        print(f"  chassis: {c['name']} → {c['base_price_gross']}")
                else:
                    for cond, info in record["prices"].items():
                        print(f"  {cond:<12} gross={info['price_gross']}  net={info['price_net']}  stock={info['stock']}  status={info['status']}")
            print(f"  ({time.perf_counter()-t0:.1f}s)\n")

    OUT_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[SAVED] {len(results)} products → {OUT_FILE.resolve()}")
    errors = [r for r in results if "error" in r]
    if errors:
        print(f"[WARN] {len(errors)} product(s) failed after all retries:")
        for e in errors:
            print(f"  {e['url']} — {e['error']}")

await test_rack()

[PLP] https://www.tonitrus.com/Rack_26?lang=eng


16:23:21  ERROR     asyncio  Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed\nCall log:\n  - navigating to "https://www.tonitrus.com/C1000-16P-E-2G-L_8?lang=eng", waiting until "domcontentloaded"\n')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Call log:
  - navigating to "https://www.tonitrus.com/C1000-16P-E-2G-L_8?lang=eng", waiting until "domcontentloaded"



[PLP] found 10 products (4 CTO)

  [CTO] https://www.tonitrus.com/Cisco-C240-M5-CTO-Server_1?lang=eng  (attempt 1)
    [CHASSIS] Cisco UCS C240 M5 - 24x SFF Server → 494,50 €
    [CHASSIS] Cisco UCS C240 M5 - 8x SFF Server → 238,50 €
  name : Cisco - Cisco_C240_M5_CTO_Server - Cisco - Cisco C240 M5 CTO Server - Cisco C240 M5 CTO Server
  brand: Cisco  mpn: Cisco_C240_M5_CTO_Server  barcode: TCO_00108-014
  chassis: Cisco UCS C240 M5 - 24x SFF Server → 494,50 €
  chassis: Cisco UCS C240 M5 - 8x SFF Server → 238,50 €
  (14.5s)

  [PDP] https://www.tonitrus.com/UCS-C220-M3SBE_5?lang=eng  (attempt 1)
  name : Cisco - UCS-C220-M3SBE - Cisco - UCS-C220-M3SBE - Cisco UCS C220 M3SBE Server 2x E5-2620V2, 32 GB, 2x 650W PSU
  brand: Cisco  mpn: UCS-C220-M3SBE  barcode: 10237052
  NEW          gross=None  net=None  stock=0  status=OOS
  REFURBISHED  gross=None  net=None  stock=0  status=OOS
  (9.3s)

  [CTO] https://www.tonitrus.com/UCS-C220-M4S_2?lang=eng  (attempt 1)
    [OOS-CTO] no chassis sw